In [10]:
import sys
from pathlib import Path

# ── Locate project root and register the basis_data package ───────────────────
# Works whether the kernel CWD is notebooks/ or the project root.
for _candidate in [Path.cwd(), Path.cwd().parent]:
    if (_candidate / "basis_data" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from basis_data.pipeline import build_bond_history

bond_history, cds_mappings_filtered = build_bond_history()
bond_history.head(5)

[cache] cds_transform_interpolated_subset_2025.parquet is current — loading from disk
cds_transform_interpolated (3729506, 14)
bond_history (raw) (119770, 5)
bond_history (final) (119770, 20)


,security,date,LAST_PRICE,Benchmark_Spread,G_Spread,MATURITY,SECURITY_NAME,target_cds_maturity,ticker,cds_index,Markit_ShortName,BOND_ISIN,bbg_cds_ticker,cds_spread,upfront,cds_spread_5y,cds_5y_maturity,cds_spread_5y_fixed,basis_spread,basis_price
0,BE6364524635 Corp,2026-07-23,98.071,66.593,65.741,2033-05-19,ABIBB 3 3/8 05/19/33,2033-06-20,ABIBB,Main,AnheuserBusch InBev,BE6364524635,ABIB CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN
1,DE000A4DFLQ6 Corp,2026-07-23,103.776,149.997,147.453,2031-04-01,SHAEFF 5 3/8 04/01/31,2031-06-20,SHAEFF,XO,Schaeffler AG,DE000A4DFLQ6,SCHAAG CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN
2,FI4000590864 Corp,2026-07-23,96.108,187.510,183.239,2031-05-28,METSA 3 7/8 05/28/31,2031-06-20,METSA,XO,Metsa Brd Corp,FI4000590864,METSA CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN
3,FR0011270487 Corp,2026-07-23,84.706,430.197,430.288,2032-06-14,FTI 4 06/14/32,2032-06-20,FTI,XO_Legacy,TechnipFMC PLC,FR0011270487,FTI CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN
4,FR0014004R72 Corp,2026-07-23,88.265,82.045,78.609,2030-07-27,ALOFP 0 1/2 07/27/30,2030-12-20,ALOFP,Main,ALSTOM,FR0014004R72,ALSTOM CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN


In [ ]:
import pandas as pd


def _last_valid(series):
    """Last non-NaN value in a series, or None if all NaN."""
    valid = series.notna()
    return series[valid].iloc[-1] if valid.any() else None


def _basis_window_stats(g, cds_col, basis_col):
    """latest/min/max basis (as a (latest, min, max) tuple) over the window
    where both cds_col and G_Spread are available -- generic helper shared by
    the target-maturity-matched basis and the fixed-5Y-maturity basis."""
    valid = g[cds_col].notna() & g["G_Spread"].notna()
    valid_dates = g.loc[valid, "date"]
    if valid_dates.empty:
        return None, None, None
    date_end = valid_dates.iloc[-1]
    latest_basis = g.loc[g["date"] == date_end, basis_col].iloc[-1]
    basis_valid = g.loc[valid, basis_col]
    return latest_basis, basis_valid.min(), basis_valid.max()


def _summarize_ticker(g):
    """One row of cash-vs-CDS basis summary stats for a single CDS ticker's
    proxy bond. date_start/date_end bound the window where both the bond's
    G-spread and the maturity-matched CDS spread are available, since basis
    is only computable there; latest_CDS/latest_bond_g/latest_basis/min_basis/
    max_basis/range_pct are all computed over that same window. The _5Y
    columns are the same stats but for the bond's G-spread vs. the CDS spread
    at the fixed 5Y maturity (cds_spread_5y_fixed) instead of the bond's own
    target maturity, since 5Y is the reliably liquid point. latest_5y and
    latest_benchmark are informational reference stats (like the equivalent
    chart legend entries), so they're just the last available value in the
    ticker's full history, independent of either basis window. range_pct is
    the latest basis's position within [min_basis, max_basis]: 100% at the
    historical max, 0% at the min."""
    ticker = g.name  # captured before sort_values, which drops the groupby name
    g = g.sort_values("date")
    valid = g["cds_spread"].notna() & g["G_Spread"].notna()
    valid_dates = g.loc[valid, "date"]

    if valid_dates.empty:
        date_start = date_end = pd.NaT
        latest_cds_spread = latest_bond_g = None
    else:
        date_start = valid_dates.iloc[0]
        date_end = valid_dates.iloc[-1]
        last_row = g.loc[g["date"] == date_end].iloc[-1]
        latest_cds_spread = last_row["cds_spread"]
        latest_bond_g = last_row["G_Spread"]

    latest_basis, min_basis, max_basis = _basis_window_stats(g, "cds_spread", "basis_spread")
    range_pct = None
    if latest_basis is not None:
        basis_range = max_basis - min_basis
        range_pct = 100.0 if basis_range == 0 else (latest_basis - min_basis) / basis_range * 100

    latest_basis_5y, min_basis_5y, max_basis_5y = _basis_window_stats(g, "cds_spread_5y_fixed", "basis_spread_5y_fixed")
    range_pct_5y = None
    if latest_basis_5y is not None:
        basis_5y_range = max_basis_5y - min_basis_5y
        range_pct_5y = 100.0 if basis_5y_range == 0 else (latest_basis_5y - min_basis_5y) / basis_5y_range * 100

    latest_5y = _last_valid(g["cds_spread_5y"]) if "cds_spread_5y" in g.columns else None
    latest_benchmark = _last_valid(g["Benchmark_Spread"]) if "Benchmark_Spread" in g.columns else None

    return pd.Series({
        "CDS_Ticker": ticker,
        "CDS_Index": g["cds_index"].iloc[0],
        "bond_name": g["SECURITY_NAME"].iloc[0],
        # grouped/hidden block: identifiers and dates, not needed at a glance
        "bbg_cds_ticker": g["bbg_cds_ticker"].iloc[0],
        "bond_ISIN": g["BOND_ISIN"].iloc[0],
        "date_start": date_start,
        "date_end": date_end,
        "cds_mat": g["target_cds_maturity"].iloc[0],
        "latest_CDS": latest_cds_spread,
        "latest_5y": latest_5y,
        "latest_bond_g": latest_bond_g,
        "latest_benchmark": latest_benchmark,
        "latest_basis": latest_basis,
        "min_basis": min_basis,
        "max_basis": max_basis,
        "range_pct": range_pct,
        "latest_basis_5Y": latest_basis_5y,
        "min_basis_5Y": min_basis_5y,
        "max_basis_5Y": max_basis_5y,
        "range_pct_5Y": range_pct_5y,
    })


basis_summary = (
    bond_history
    .groupby("ticker")
    .apply(_summarize_ticker)
    .reset_index(drop=True)
    .sort_values(["CDS_Index", "CDS_Ticker"])
    .reset_index(drop=True)
)
print(basis_summary.shape)
basis_summary.head(10)

In [ ]:
import os

from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

OUTPUT_PATH = Path(r"S:\Structured Credit\Matt Stuff\claude_repos\cds_bond_basis\output\bond_cds_basis_summary.xlsx")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Bloomberg field mnemonic for the bond G-spread, kept consistent with the
# field already pulled via bdh_mm() into bond_history's G_Spread column.
G_SPREAD_FIELD = "BLOOMBERG_MID_G_SPREAD"

FORMULA_FILL = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")

RANGE_PCT_COLOR_SCALE = dict(
    start_type="num", start_value=0, start_color="F8696B",
    mid_type="num", mid_value=50, mid_color="FFFFFF",
    end_type="num", end_value=100, end_color="5A8AC6",
)

# Identifier/date/reference columns, grouped and collapsed since they're rarely needed at a glance.
GROUPED_COLUMNS = [
    "bbg_cds_ticker", "bond_ISIN", "date_start", "date_end", "cds_mat",
    "latest_CDS", "latest_5y", "latest_bond_g", "latest_benchmark",
]
# Separate group: intermediate live lookups feeding spread_live/g_live, not
# usually needed on their own.
LIVE_GROUPED_COLUMNS = {"bond_bench", "bench_price"}


def _iferror(formula, fallback='"--"'):
    """Wrap a '=...' formula string as =IFERROR(..., fallback)."""
    return f"=IFERROR({formula[1:]},{fallback})"


COLUMN_WIDTHS = {
    "CDS_Ticker": 12, "bbg_cds_ticker": 26, "CDS_Index": 10, "bond_name": 28, "bond_ISIN": 16,
    "date_start": 12, "date_end": 12, "cds_mat": 12,
    "latest_CDS": 14, "latest_5y": 12, "latest_bond_g": 14, "latest_benchmark": 16,
    "latest_basis": 14, "min_basis": 12, "max_basis": 12, "range_pct": 12,
    "latest_basis_5Y": 14, "min_basis_5Y": 12, "max_basis_5Y": 12, "range_pct_5Y": 12,
}
DATE_COLUMNS = {"date_start", "date_end", "cds_mat"}
SPREAD_COLUMNS = {
    "latest_CDS", "latest_5y", "latest_bond_g", "latest_benchmark",
    "latest_basis", "min_basis", "max_basis",
    "latest_basis_5Y", "min_basis_5Y", "max_basis_5Y",
}
PERCENT_COLUMNS = {"range_pct", "range_pct_5Y"}

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    basis_summary.to_excel(writer, sheet_name="Basis Summary", index=False)
    ws = writer.sheets["Basis Summary"]

    for cell in ws[1]:
        cell.font = Font(bold=True)
    ws.freeze_panes = "D2"

    last_row = len(basis_summary) + 1
    for i, col in enumerate(basis_summary.columns, start=1):
        col_letter = get_column_letter(i)
        ws.column_dimensions[col_letter].width = COLUMN_WIDTHS.get(col, 14)
        if col in GROUPED_COLUMNS:
            ws.column_dimensions[col_letter].outlineLevel = 1
            ws.column_dimensions[col_letter].hidden = True
        if col in DATE_COLUMNS:
            for cell in ws[col_letter][1:]:
                cell.number_format = "yyyy-mm-dd"
        elif col in SPREAD_COLUMNS:
            # accounting style: negatives in red parens, matching the slideshow charts
            for cell in ws[col_letter][1:]:
                cell.number_format = "#,##0.0;[Red](#,##0.0)"
        elif col in PERCENT_COLUMNS:
            for cell in ws[col_letter][1:]:
                cell.number_format = "0.0\\%"
            # range_pct is already normalized 0-100, so anchor the color scale to
            # those fixed values (not the observed min/max in the column) --
            # red at 0 (historical low), white at 50, blue at 100 (historical high)
            ws.conditional_formatting.add(
                f"{col_letter}2:{col_letter}{last_row}", ColorScaleRule(**RANGE_PCT_COLOR_SCALE)
            )

    # Static columns referenced by the live formulas below.
    isin_col = get_column_letter(basis_summary.columns.get_loc("bond_ISIN") + 1)
    cds_ticker_col = get_column_letter(basis_summary.columns.get_loc("bbg_cds_ticker") + 1)
    # live_basis is a bond-G-spread-vs-5Y-CDS basis, so its range check uses
    # the fixed-5Y-maturity historical range, not the target-maturity one.
    min_basis_col = get_column_letter(basis_summary.columns.get_loc("min_basis_5Y") + 1)
    max_basis_col = get_column_letter(basis_summary.columns.get_loc("max_basis_5Y") + 1)

    # Live-refresh columns: re-derive the bond's benchmark, its G-spread, and
    # the CDS-vs-bond basis directly in Excel via the Bloomberg Add-in (BDP),
    # rather than from the historical Python pull. Requires the Add-in to
    # resolve. Each column's formula references the live column(s) to its left.
    # Every formula is wrapped in IFERROR(..., "--") so a missing/unresolved
    # BDP lookup shows as a placeholder instead of an #N/A propagating
    # through the dependent formulas to its right.
    live_col_start = len(basis_summary.columns) + 1
    live_columns = [
        ("bond_live", 14, "#,##0.000"),
        ("bond_bench", 22, None),
        ("bench_price", 14, "#,##0.000"),
        ("spread_live", 14, "#,##0.0;[Red](#,##0.0)"),
        ("g_live", 14, "#,##0.0;[Red](#,##0.0)"),
        ("live_cds_5y", 14, "#,##0.0;[Red](#,##0.0)"),
        ("live_basis", 14, "#,##0.0;[Red](#,##0.0)"),
        ("live_range_pct", 14, "0.0\\%"),
    ]
    (bond_live_col, bond_bench_col, bench_price_col, spread_live_col,
     g_live_col, live_cds_5y_col, live_basis_col, live_range_pct_col) = (
        get_column_letter(live_col_start + offset) for offset in range(len(live_columns))
    )

    for offset, (name, width, _) in enumerate(live_columns):
        col_letter = get_column_letter(live_col_start + offset)
        ws.cell(row=1, column=live_col_start + offset, value=name).font = Font(bold=True)
        ws.column_dimensions[col_letter].width = width
        if name in LIVE_GROUPED_COLUMNS:
            ws.column_dimensions[col_letter].outlineLevel = 1
            ws.column_dimensions[col_letter].hidden = True

    for row in range(2, last_row + 1):
        ws[f"{bond_live_col}{row}"] = _iferror(f'=BDP({isin_col}{row}&" Corp","LAST_PRICE")')
        ws[f"{bond_bench_col}{row}"] = _iferror(f'=BDP({isin_col}{row}&" Corp","YAS_BENCHMARK_BOND_ISIN")')
        # bond_bench is a bare ISIN (from YAS_BENCHMARK_BOND_ISIN), so it needs
        # the same " CORP" suffix as bond_ISIN to resolve as a BDP security.
        ws[f"{bench_price_col}{row}"] = _iferror(f'=BDP({bond_bench_col}{row}&" CORP","LAST_PRICE")')
        ws[f"{spread_live_col}{row}"] = _iferror(
            f'=BDP({isin_col}{row}&" Corp","YAS_YLD_SPREAD",'
            f'"YAS_BOND_PX="&{bond_live_col}{row},"YAS_BNCHMRK_BOND_PX="&{bench_price_col}{row})'
        )
        ws[f"{g_live_col}{row}"] = _iferror(
            f'=BDP({isin_col}{row}&" Corp","{G_SPREAD_FIELD}",'
            f'"YAS_BOND_PX="&{bond_live_col}{row},"YAS_BNCHMRK_BOND_PX="&{bench_price_col}{row})'
        )
        ws[f"{live_cds_5y_col}{row}"] = _iferror(f'=BDP({cds_ticker_col}{row},"LAST_PRICE")')
        ws[f"{live_basis_col}{row}"] = _iferror(f'={live_cds_5y_col}{row}-{g_live_col}{row}')
        ws[f"{live_range_pct_col}{row}"] = _iferror(
            f'=({live_basis_col}{row}-{min_basis_col}{row})'
            f'/({max_basis_col}{row}-{min_basis_col}{row})*100'
        )

    for offset, (name, width, number_format) in enumerate(live_columns):
        col_letter = get_column_letter(live_col_start + offset)
        # light grey fill flags these as formula-driven (live BDP pulls), unlike
        # the plain data columns to their left
        for cell in ws[col_letter][1:]:
            cell.fill = FORMULA_FILL
            if number_format is not None:
                cell.number_format = number_format

    # Same fixed 0/50/100 red-white-blue scale as range_pct, applied to its live counterpart.
    ws.conditional_formatting.add(
        f"{live_range_pct_col}2:{live_range_pct_col}{last_row}", ColorScaleRule(**RANGE_PCT_COLOR_SCALE)
    )

    # Filter dropdowns on the header row, spanning every column (static + live).
    last_col_letter = get_column_letter(live_col_start + len(live_columns) - 1)
    ws.auto_filter.ref = f"A1:{last_col_letter}{last_row}"

print(f"Wrote {len(basis_summary)} rows to {OUTPUT_PATH}")
os.startfile(OUTPUT_PATH)